# Fine-tune `intfloat/multilingual-e5-base` — 1 epoch

Notebook này fine-tune model `intfloat/multilingual-e5-base` trên bộ JSONL đã chuẩn bị sẵn:
- `data/training/train_5000.jsonl`
- `data/training/valid_5000.jsonl`
- `data/training/test_5000.jsonl`

Mục tiêu: train 1 epoch cho bài toán semantic search / retrieval sản phẩm.

Output model sẽ được lưu trong `embedding_project/models/e5_base_finetuned_1epoch/`.

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" datasets pandas scikit-learn numpy tqdm torch

## 2) Thiết lập đường dẫn và kiểm tra GPU

In [ ]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import ndcg_score
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

WORKSPACE = Path('/content/llm_provider_benchmarking')
DATA_DIR = WORKSPACE / 'data' / 'training'
MODEL_DIR = WORKSPACE / 'embedding_project' / 'models' / 'e5_base_finetuned_1epoch'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train_5000.jsonl'
VALID_PATH = DATA_DIR / 'valid_5000.jsonl'
TEST_PATH = DATA_DIR / 'test_5000.jsonl'

BASE_MODEL = 'intfloat/multilingual-e5-base'
MAX_SEQ_LENGTH = 512
EPOCHS = 1
BATCH_SIZE = 8 if torch.cuda.is_available() else 2
FP16 = torch.cuda.is_available()
LR = 2e-5

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Train file:', TRAIN_PATH)
print('Valid file:', VALID_PATH)
print('Test file :', TEST_PATH)
print('Model dir  :', MODEL_DIR)

## 3) Load JSONL dataset

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
valid_rows = load_jsonl(VALID_PATH)
test_rows = load_jsonl(TEST_PATH)

print('train rows:', len(train_rows))
print('valid rows:', len(valid_rows))
print('test rows :', len(test_rows))

def to_sentence_transformers_rows(rows):
    out = []
    for r in rows:
        q = str(r.get('query', '')).strip()
        p = str(r.get('positive', '')).strip()
        if q and p:
            out.append({'anchor': q, 'positive': p})
    return out

train_ds = Dataset.from_list(to_sentence_transformers_rows(train_rows))
valid_ds = Dataset.from_list(to_sentence_transformers_rows(valid_rows))
print(train_ds)
print(valid_ds)

## 4) Fine-tune E5-base trong 1 epoch

Dùng `MultipleNegativesRankingLoss` cho positive pairs. Các mẫu khác trong batch sẽ đóng vai trò negative ngầm.

In [ ]:
model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_SEQ_LENGTH
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    fp16=FP16,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=25,
    save_total_limit=2,
    report_to=[],
    run_name='e5-base-vn-1epoch',
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=loss,
)

trainer.train()
model.save(str(MODEL_DIR))
print('Saved model to:', MODEL_DIR)

## 5) Đánh giá nhanh trên test set

Notebook dùng retrieval đơn giản: lấy embedding query và corpus, tính Recall@10, MRR@10, NDCG@10.
Nếu bạn đã có ground-truth mapping query → relevant products thì có thể thay phần labels ở dưới bằng file nhãn thật của bạn.

In [ ]:
def normalize_embeddings(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def embed_texts(texts, prefix):
    inputs = [f'{prefix}: {t}' for t in texts]
    emb = model.encode(inputs, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)
    return normalize_embeddings(emb)

test_queries = []
test_positive_texts = []
for r in test_rows:
    q = str(r.get('query', '')).strip()
    p = str(r.get('positive', '')).strip()
    if q and p:
        test_queries.append(q)
        test_positive_texts.append(p)

query_emb = embed_texts(test_queries, 'query')
passage_emb = embed_texts(test_positive_texts, 'passage')

scores = query_emb @ passage_emb.T
top10 = np.argsort(-scores, axis=1)[:, :10]

# Vì test file ở đây là positive pair theo từng dòng, ta đánh giá retrieval nội bộ bằng cách xem
# liệu đúng positive của query có nằm trong top-10 hay không.
hits_at_10 = []
rr_at_10 = []
ndcg_at_10 = []

for i in range(len(test_queries)):
    retrieved = top10[i].tolist()
    hit = 1 if i in retrieved else 0
    hits_at_10.append(hit)
    if i in retrieved:
        rank = retrieved.index(i) + 1
        rr_at_10.append(1.0 / rank)
        rel = np.zeros(10)
        rel[rank - 1] = 1
        ndcg_at_10.append(ndcg_score([rel], [rel]))
    else:
        rr_at_10.append(0.0)
        ndcg_at_10.append(0.0)

print('Recall@10 (proxy):', float(np.mean(hits_at_10)))
print('MRR@10    (proxy):', float(np.mean(rr_at_10)))
print('NDCG@10   (proxy):', float(np.mean(ndcg_at_10)))

print('\nSample query:')
print(test_queries[0])
print('\nPositive text:')
print(test_positive_texts[0][:1200])

## 6) Lưu ý cho E5

Khi dùng model sau fine-tune, nên encode theo prefix đúng chuẩn E5:
- query: `query: ...`
- passage: `passage: ...`

Nếu bạn muốn, bước tiếp theo mình có thể tạo thêm notebook riêng để:
- build index với FAISS / Qdrant
- evaluate retrieval chuẩn hơn theo tập nhãn ground-truth
- hoặc fine-tune tiếp 2/3 epochs.